# 🔥 TERRA-FIRE: Data Engineering Lab Pipeline
### **Materia:** Data Science (LITD9A) — Universidad Tecnológica El Retoño
### **Docente Asesor:** Dr. Sergio Valadez Godínez
### **Autores:** Angel Emilio Gutierrez Lozano & Emiliano Betanzos Valtierra
### **Proyecto:** TERRA-FIRE — Plataforma de Inteligencia Predictiva de Incendios Forestales (TRL 2/3)

---

## 🎯 Propósito del Laboratorio
Este notebook implementa el pipeline de **Ingeniería de Datos (ETL)** reproducible de extremo a extremo para el proyecto **TERRA-FIRE**. Transiciona las fuentes adquiridas en la actividad previa hacia un **Dataset Listo para Análisis (*Analysis-Ready Dataset*)** en formato columnar **Apache Parquet**, aplicando validaciones rigurosas de integridad, transformaciones termodinámicas (Vapor Pressure Deficit vía ecuación de Tetens) y armonización espaciotemporal a resolución microterritorial.

## 📦 1. Configuración del Entorno y Carga de Dependencias

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Añadir directorio raíz al path para importar módulos de src/
sys.path.append(os.path.abspath('..'))

from src.extract import extract_era5_hourly, extract_nasa_firms, extract_conafor_fires, extract_sentinel2_stac, profile_dataframe
from src.clean import clean_era5_data, clean_firms_data, clean_conafor_data
from src.transform import calculate_tetens_vpd, decompose_wind_vectors, calculate_lapse_rate_temperature, transform_climate_daily_aggregates
from src.integrate import integrate_spatiotemporal_matrix, export_analysis_ready_parquet, build_regional_monitoring_grid
from src.validate import run_data_validation_suite

print("✅ Módulos del pipeline TERRA-FIRE cargados exitosamente.")

## 🔍 2. Fase 1: Extracción y Perfilado de Datos Crudos (Profiling)
Extraemos las observaciones desde el almacenamiento institucional (`TERRAFIRE-db` / `data/raw`) y diagnosticamos dimensiones, nulos y tipos.

In [ ]:
# 1. Extracción de Reanálisis ERA5-Land Horario (Sierra Fría 2023-2024)
df_era5_raw, era5_meta = extract_era5_hourly()
print(f"ERA5 Metadatos de Estación: {era5_meta}")
prof_era5 = profile_dataframe("ECMWF ERA5-Land", df_era5_raw)
print(f"Dimensiones ERA5: {prof_era5['total_records']} filas x {prof_era5['total_variables']} columnas | Nulos: {prof_era5['columns_with_nulls']}")

# 2. Extracción de Detecciones Térmicas NASA FIRMS VIIRS (2023–2024)
df_firms23 = extract_nasa_firms(2023)
df_firms24 = extract_nasa_firms(2024)
df_firms_raw = pd.concat([df_firms23, df_firms24], ignore_index=True)
prof_firms = profile_dataframe("NASA FIRMS VIIRS", df_firms_raw)
print(f"Dimensiones NASA FIRMS: {prof_firms['total_records']} detecciones | Columnas: {prof_firms['total_variables']}")

# 3. Extracción de Incendios Históricos CONAFOR (2012–2025)
df_conafor_raw = extract_conafor_fires()
prof_conafor = profile_dataframe("CONAFOR Histórico", df_conafor_raw)
print(f"Dimensiones CONAFOR: {prof_conafor['total_records']} registros oficiales | Columnas: {prof_conafor['total_variables']}")

## 🧹 3. Fase 2: Limpieza y Rectificación de Calidad
Aplicamos reglas algorítmicas transparentes para corregir encabezados con caracteres especiales, imputar nulos en estratos forestales, corregir signos de longitud geográfica y unificar marcas temporales.

In [ ]:
# Limpieza de datos meteorológicos
df_era5_clean, log_era5 = clean_era5_data(df_era5_raw)

# Limpieza y filtrado regional de NASA FIRMS para Aguascalientes / Sierra Fría
bbox_ags = {'min_lat': 21.5, 'max_lat': 22.6, 'min_lon': -103.1, 'max_lon': -101.8}
df_firms_clean, log_firms = clean_firms_data(df_firms_raw, bbox=bbox_ags)

# Limpieza y filtrado de CONAFOR para Aguascalientes
df_conafor_clean, log_conafor = clean_conafor_data(df_conafor_raw, target_state='Aguascalientes')

print("Bitácora de Limpieza de ERA5:")
print(pd.DataFrame(log_era5)[['issue', 'records_affected', 'result']])

## ⚗️ 4. Fase 3: Transformación Biofísica y Fórmulas Físicas
Calculamos variables analíticas avanzadas requeridas por la física del fuego:
1. **Déficit de Presión de Vapor ($VPD$ en $kPa$)** mediante la formulación empírica de Tetens.
2. **Descomposición vectorial del viento** en componentes zonal ($u_{10}$) y meridional ($v_{10}$).
3. **Reducción adiabática ambiental (*Lapse-rate Downscaling*)** sobre relieve DEM.

In [ ]:
# Demostración del cálculo de VPD mediante ecuación de Tetens
sample_temp = pd.Series([15.0, 25.0, 32.0, 38.0])
sample_dew = pd.Series([10.0, 8.0, 4.0, 0.0])
vpd_calc = calculate_tetens_vpd(sample_temp, sample_dew)

demo_df = pd.DataFrame({
    'Temperatura (°C)': sample_temp,
    'Punto Rocío (°C)': sample_dew,
    'VPD Calculado (kPa)': vpd_calc
})
print("Demostración de Ecuación de Tetens:")
display(demo_df)

## 🌐 5. Fase 4: Integración Espaciotemporal y Formulación de la Variable Objetivo
Fusionamos las variables físicas sobre la malla regional de 25 nodos de monitoreo en Aguascalientes / Sierra Fría a lo largo de los 731 días de 2023 y 2024 (18,275 observaciones vóxel-tiempo), formulando la etiqueta de supervisión `fire_ignition_48h` con 48–72h de anticipación.

In [ ]:
matrix_df, int_record = integrate_spatiotemporal_matrix(
    df_era5_clean=df_era5_clean,
    df_firms_clean=df_firms_clean,
    df_conafor_clean=df_conafor_clean,
    matching_radius_deg=0.08
)

print("Registro de Integración Espaciotemporal:")
for k, v in int_record.items():
    print(f"  {k}: {v}")

print("\nPrimeros registros de la matriz integrada:")
display(matrix_df.head(5))

## 💾 6. Fase 5: Almacenamiento Analítico Optimizado (Apache Parquet)
Exportamos el dataset resultante en formato Apache Parquet con compresión Snappy y comparamos métricas de tamaño y rendimiento respecto al formato CSV plano.

In [ ]:
parquet_output = '../data/processed/terra_fire_spatiotemporal_matrix_v1.parquet'
sample_csv_output = '../data/sample/terra_fire_matrix_sample.csv'

metrics = export_analysis_ready_parquet(
    df=matrix_df,
    output_parquet_path=parquet_output,
    output_sample_csv_path=sample_csv_output
)

print("Métricas de Almacenamiento:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## ✅ 7. Fase 6: Suite Automatizada de Validación de Calidad
Ejecutamos el checklist de 7 puntos exigido en el paso 7 de la actividad para certificar que el pipeline no introdujo pérdidas ni anomalías.

In [ ]:
all_passed, checklist = run_data_validation_suite(parquet_output)
chk_df = pd.DataFrame(checklist)

print(f"Resultado General de Validación: {'TODAS LAS PRUEBAS APROBADAS' if all_passed else 'FALLOS DETECTADOS'}")
display(chk_df[['status', 'validation_check', 'expected_condition', 'result']])

## 🏆 8. Dictamen Final de Disponibilidad de Datos (*Data Readiness Decision*)

> [!IMPORTANT]
> **DICTAMEN OFICIAL: READY FOR ANALYSIS**
>
> El dataset procesado `terra_fire_spatiotemporal_matrix_v1.parquet` cumple al 100% con los criterios de integridad estructural, unicidad de llaves, límites termodinámicos válidos, ausencia de valores nulos y trazabilidad completa hacia sus fuentes satelitales e institucionales. Queda certificado y disponible para las etapas subsiguientes de **Exploratory Data Analysis (EDA)**, ingeniería de características para ML y entrenamiento de modelos predictivos anticipatorios.